In [1]:
from datasets import load_dataset
import tqdm as notebook_tqdm

C:\Users\casvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# dataset_name = "squad_v2"
# dataset = load_dataset(dataset_name, split="train")
# eval_dataset = load_dataset(dataset_name, split="validation")
# print("dataset: ",dataset)
# print("eval_dataset: ",eval_dataset)

In [3]:
dataset_path="dataset/small_datset.jsonl"
dataset_name = "tssb_data_3M"
dataset = load_dataset("json", data_files=dataset_path, split="train[:90%]")
eval_dataset = load_dataset("json", data_files=dataset_path, split="train[90%:]")

print("dataset: ",dataset)
print("eval_dataset: ",eval_dataset)

dataset:  Dataset({
    features: ['prompt', 'completion'],
    num_rows: 31500
})
eval_dataset:  Dataset({
    features: ['prompt', 'completion'],
    num_rows: 3500
})


In [4]:
import torch
cuda_available = torch.cuda.is_available()

if cuda_available:
    device_id = 0  # You can change to 1,2,3 if you want other GPUs
    torch.cuda.set_device(device_id)
    # device = torch.device(f"cuda:{device_id}")
    device = torch.device(f"cuda:{device_id}")
    print(f"🖥️ Using GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
else:
    device = torch.device("cpu")
    print("⚙️ No GPU available, using CPU.")

print(f"Device selected: {device}")

🖥️ Using GPU 0: NVIDIA GeForce RTX 4070 SUPER
Device selected: cuda:0


In [5]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = 4096,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.4.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 2. Max memory: 11.994 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:22<00:00, 11.49s/it]


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2025.4.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [7]:
import neptune
import neptune.integrations.optuna as optuna_utils
run = neptune.init_run(
    project="casvi/CodeMedic",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIzMTMzYjhhOC1jYzA1LTQ0YjAtOTJjNi1iY2EzM2VhMDY0OTcifQ=="
)


[neptune] [warning] NeptuneWarning: By default, these monitoring options are disabled in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', 'capture_hardware_metrics'. You can set them to 'True' when initializing the run and the monitoring will continue until you call run.stop() or the kernel stops. NOTE: To track the source files, pass their paths to the 'source_code' argument. For help, see: https://docs-legacy.neptune.ai/logging/source_code/


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/casvi/CodeMedic/e/COD-65


In [8]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
acc = evaluate.load("accuracy")

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = labels[:, 1:]
    preds = preds[:, :-1]

    mask = labels == -100
    labels[mask] = tokenizer.pad_token_id
    preds[mask] = tokenizer.pad_token_id

    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    bleu_score = bleu.compute(predictions=decoded_preds, references=decoded_labels)
    accuracy = acc.compute(predictions=preds[~mask], references=labels[~mask])
    rouge_score = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        **bleu_score,
        **accuracy,
        **rouge_score
    }

In [9]:
# from trl import SFTTrainer, SFTConfig
# import time
# start=time.time()
#
# # SFT Config
# config = SFTConfig(
#     dataset_num_proc = 1,
#     #output_dir="./outputs",
#     dataset_text_field="prompt",#Depends on the colum of your data set
#     #dataset_text_field="question",#Depends on the colum of your data set
#     # learning_rate=1e-5,
#     learning_rate=2e-4,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=1,
#     num_train_epochs=5,
#     report_to="none",
#     logging_steps=100,
#     max_steps=2000,
#     eval_accumulation_steps=100,
# )
# trainer = SFTTrainer(
#     model=model,  # base or PEFT model
#     tokenizer=tokenizer,
#     train_dataset=dataset,
#     eval_dataset=eval_dataset,
#     args=config,
#     warmup_steps = 5,
#     weight_decay = 0.01,
#     compute_metrics = compute_metrics,
#     preprocess_logits_for_metrics=preprocess_logits_for_metrics,
# )
# metrics = trainer.evaluate()
# print("Metrics:",metrics)
# trainer.train()
#
#
# end = time.time()
# length = end - start
#
# hours = int(length // 3600)
# minutes = int((length % 3600) // 60)
# seconds = int(length % 60)
#
# print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")


In [10]:
# metrics = trainer.evaluate()
# print("Metrics:",metrics)


In [11]:
from trl import SFTTrainer, SFTConfig
import time
def objective(trial):
    start=time.time()
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5,5e-4, log=True)
    # batch_size = trial.suggest_categorical("per_device_train_batch_size", [4,6,8])
    # gradient_accumulation_steps = trial.suggest_categorical("gradient_accumulation_steps",[4,6,8])
    num_epochs = trial.suggest_int("num_train_epochs", 5, 10)
    batch_size=1

    # SFT Config
    config = SFTConfig(
        dataset_num_proc = 1,
        #output_dir="./outputs",
        dataset_text_field="prompt",#Depends on the colum of your data set
        #dataset_text_field="question",#Depends on the colum of your data set
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=batch_size,
        num_train_epochs=num_epochs,
        report_to="none",
        logging_steps=100,
        max_steps=1000,
        eval_accumulation_steps=100
    )

    trainer = SFTTrainer(
        model=model,  # base or PEFT model
        tokenizer=tokenizer,
        train_dataset=dataset,
        eval_dataset=eval_dataset,
        args=config,
        warmup_steps = 5,
        weight_decay = 0.01,
        compute_metrics = compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()
    print("Metrics:",metrics)

    # Log trial info to Neptune
    run[f"trial/{trial.number}/metrics"] = metrics
    run[f"trial/{trial.number}/params"] = {
        "learning_rate": learning_rate,
        "num_epochs": num_epochs,
    }


    end = time.time()
    length = end - start

    hours = int(length // 3600)
    minutes = int((length % 3600) // 60)
    seconds = int(length % 60)

    print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")

    return metrics["eval_loss"]  # Or any other metric


In [ ]:
import optuna
neptune_callback = optuna_utils.NeptuneCallback(run)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10, callbacks=[neptune_callback], show_progress_bar=True)

[I 2025-05-12 23:53:17,880] A new study created in memory with name: no-name-061b4938-7467-409f-ae9f-be4cafec12a9
  0%|          | 0/10 [00:00<?, ?it/s]==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31,500 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 174,587,904/8,000,000,000 (2.18% trained)


Step,Training Loss
100,1.743400
200,1.379600
300,1.342400
400,1.285600
500,1.308400
600,1.278700
700,1.282000
800,1.330000
900,1.255100
1000,1.303300


Unsloth: Will smartly offload gradients to save VRAM!


[neptune] [warning] NeptuneUnsupportedType: You're attempting to log a type that is not directly supported by Neptune (<class 'list'>).
        Convert the value to a supported type, such as a string or float, or use stringify_unsupported(obj)
        for dictionaries or collections that contain unsupported values.
        For more, see https://docs-legacy.neptune.ai/help/value_of_unsupported_type
  0%|          | 0/10 [12:29<?, ?it/s]

Metrics: {'eval_loss': 1.3014625310897827, 'eval_bleu': 0.583318757435576, 'eval_precisions': [0.804586583081312, 0.6361372821340541, 0.5362209458508692, 0.4525404655666488], 'eval_brevity_penalty': 0.9825954183445175, 'eval_length_ratio': 0.9827451357888024, 'eval_translation_length': 163346, 'eval_reference_length': 166214, 'eval_accuracy': 0.7483742383005371, 'eval_rouge1': 0.7229803185474536, 'eval_rouge2': 0.5321052689348771, 'eval_rougeL': 0.7087076058081919, 'eval_rougeLsum': 0.7084352272415883, 'eval_runtime': 106.5966, 'eval_samples_per_second': 32.834, 'eval_steps_per_second': 4.109}
It took 0 hours, 12 minutes, and 29 seconds to train the model!
[I 2025-05-13 00:05:47,646] Trial 0 finished with value: 1.3014625310897827 and parameters: {'learning_rate': 1.5800206495535333e-05, 'num_train_epochs': 7}. Best is trial 0 with value: 1.3014625310897827.


  0%|          | 0/10 [12:30<?, ?it/s]

[W 2025-05-13 00:05:48,477] Param learning_rate unique value length is less than 2.


Unsloth: Tokenizing ["prompt"]: 100%|██████████| 31500/31500 [00:02<00:00, 14556.96 examples/s]

Unsloth: Tokenizing ["prompt"]: 100%|██████████| 3500/3500 [00:00<00:00, 7829.47 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31,500 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 174,587,904/8,000,000,000 (2.18% trained)


Step,Training Loss
100,1.111500
200,1.057800
300,1.066900
400,1.036400
500,1.086800
600,1.092800
700,1.118000
800,1.222200


In [13]:
# Get the best parameters
best_trial = study.best_trial

best_params = best_trial.params
print("best_params: ",best_params)

best_value = best_trial.value
print("Eval loss:", best_value)
run.stop()